# A.5: Full fine-tuning

Train the base model on the synthetic data with AdamW, warmup, and linear decay, with and without the prompt mask.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 A synthetic instruction dataset
import random, math, time

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

WORDS = ("apple river stone cloud garden window candle forest pencil mirror ocean silver "
         "basket rocket island winter summer planet dragon castle butter meadow violin "
         "tiger lemon piano wallet bridge jacket carpet rabbit button harbor lantern "
         "orange parrot saddle ticket valley wizard anchor bottle circle desert engine "
         "feather guitar hammer jungle kettle ladder marble needle pepper puzzle ribbon "
         "shadow thunder tunnel velvet walnut yellow zipper blanket cactus dolphin").split()

def sentence(rng):
    return " ".join(rng.sample(WORDS, rng.randint(3, 6)))

TASKS = {  # name: (phrasings; the last one is held out, answer function, input maker)
    "upper": (["Convert to uppercase: {x}", "Write {x} in capital letters.",
               "Uppercase this word: {x}", "Rewrite the word {x} using only capital letters."],
              lambda x: x.upper(), lambda r: r.choice(WORDS)),
    "first": (["What is the first letter of '{x}'?", "Give the first letter of the word {x}.",
               "Which letter does '{x}' start with?", "Tell me the initial letter of {x}."],
              lambda x: x[0], lambda r: r.choice(WORDS)),
    "repeat": (["Repeat the word {x} three times.", "Say {x} three times, separated by spaces.",
                "Write '{x}' 3 times.", "Output the word {x} thrice."],
               lambda x: " ".join([x] * 3), lambda r: r.choice(WORDS)),
    "count": (["How many words are in this text: {x}", "Count the words: {x}",
               "Number of words in '{x}'?", "Tell me how many words this has: {x}"],
              lambda x: str(len(x.split())), sentence),
    "last": (["What is the last word of: {x}", "Give the final word of '{x}'.",
              "Which word comes last in: {x}", "Name the word at the end of: {x}"],
             lambda x: x.split()[-1], sentence),
    "add": (["What is {a} + {b}?", "Add {a} and {b}.", "Compute {a} plus {b}.",
             "What do you get when you add {a} to {b}?"],
            None, lambda r: (r.randint(10, 49), r.randint(1, 9))),
}

def make_example(task, rng, held_out=False):
    phrasings, answer, make = TASKS[task]
    p = phrasings[-1] if held_out else rng.choice(phrasings[:-1])
    x = make(rng)
    if task == "add":
        return p.format(a=x[0], b=x[1]), str(x[0] + x[1])
    return p.format(x=x), answer(x)

rng = random.Random(0)

train = [make_example(t, rng) for _ in range(100) for t in TASKS]

test_seen = [make_example(t, rng) for _ in range(5) for t in TASKS]

test_new = [make_example(t, rng, held_out=True) for _ in range(5) for t in TASKS]

print(len(train), "training examples,", len(test_seen), "test (seen phrasings),",
      len(test_new), "test (held-out phrasings)")

print("seen-phrasing test examples that also occur in the training set:",
      sum(ex in set(train) for ex in test_seen))

for q, a in train[:6]:
    print(f"  {q!r} -> {a!r}")

# From A.2 Tokenizing with loss masks and padding
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

IM_START, IM_END = "<|im_start|>", "<|im_end|>"

def prompt_text(q):
    return f"{IM_START}user\n{q}{IM_END}\n{IM_START}assistant\n"

def encode(q, a, mask_prompt=True):
    p = tok(prompt_text(q), add_special_tokens=False).input_ids
    r = tok(a + IM_END, add_special_tokens=False).input_ids
    labels = ([-100] * len(p) if mask_prompt else p) + r
    return p + r, labels

def collate(batch):
    n = max(len(ids) for ids, _ in batch)
    ids = torch.full((len(batch), n), tok.pad_token_id or 0)
    labels = torch.full((len(batch), n), -100)
    attn = torch.zeros((len(batch), n), dtype=torch.long)
    for i, (x, y) in enumerate(batch):
        ids[i, :len(x)], labels[i, :len(y)], attn[i, :len(x)] = torch.tensor(x), torch.tensor(y), 1
    return ids, labels, attn

ids, labels = encode(*train[0])

print("prompt + answer tokens:", len(ids), "| trained:", sum(l != -100 for l in labels))

print("trained:", [tok.decode([t]) for t, l in zip(ids, labels) if l != -100])

# From A.4 Evaluation: exact match and forgetting
text = open("shakespeare.txt", encoding="utf-8").read()

held = tok(text[-20000:], add_special_tokens=False).input_ids

blocks = [held[i:i + 128] for i in range(0, 20 * 128, 128)]

@torch.no_grad()
def accuracy(model, data):
    model.eval()
    correct = 0
    for q, a in data:
        ids = torch.tensor([tok(prompt_text(q), add_special_tokens=False).input_ids])
        out = model.generate(ids, max_new_tokens=12, do_sample=False,
                             eos_token_id=tok.convert_tokens_to_ids(IM_END),
                             pad_token_id=tok.convert_tokens_to_ids(IM_END))
        pred = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
        correct += pred == a
    return correct / len(data)

@torch.no_grad()
def perplexity(model):
    model.eval()
    x = torch.tensor(blocks)
    return math.exp(model(x, labels=x).loss.item())

def report(name, model):
    print(f"{name:34s} seen {accuracy(model, test_seen):.2f}  held-out phrasing "
          f"{accuracy(model, test_new):.2f}  Shakespeare ppl {perplexity(model):.1f}")


In [ ]:
def finetune(mask_prompt, steps=120, batch_size=8, lr=5e-5, warmup=10, seed=0):
    torch.manual_seed(seed)
    model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M", dtype=torch.float32)
    data = [encode(q, a, mask_prompt) for q, a in train]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warmup, (steps - s) / (steps - warmup)))
    order = random.Random(seed)
    model.train()
    for step in range(steps):
        batch = order.sample(data, batch_size)
        ids, labels, attn = collate(batch)
        loss = model(input_ids=ids, attention_mask=attn, labels=labels).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        if (step + 1) % 40 == 0:
            print(f"  step {step + 1}: training loss {loss.item():.3f}")
    return model

start = time.time()
masked = finetune(mask_prompt=True)
report("fine-tuned, prompt masked", masked)
unmasked = finetune(mask_prompt=False)
report("fine-tuned, loss on all tokens", unmasked)
print(f"(both runs: {(time.time() - start) / 60:.0f} minutes on 4 CPU threads)")


**Expected output**

Output:

```text
  step 40: training loss 1.738
  step 80: training loss 0.312
  step 120: training loss 0.222
fine-tuned, prompt masked          seen 0.63  held-out phrasing 0.60  Shakespeare ppl 47.0
  step 40: training loss 1.343
  step 80: training loss 0.744
  step 120: training loss 0.658
fine-tuned, loss on all tokens     seen 0.67  held-out phrasing 0.57  Shakespeare ppl 53.0
(both runs: 12 minutes on 4 CPU threads)
```


In [ ]:
for q, a in test_new[::5]:
    ids = torch.tensor([tok(prompt_text(q), add_special_tokens=False).input_ids])
    with torch.no_grad():
        out = masked.generate(ids, max_new_tokens=12, do_sample=False,
                              eos_token_id=tok.convert_tokens_to_ids(IM_END),
                              pad_token_id=tok.convert_tokens_to_ids(IM_END))
    print(f"{q!r}: {tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)!r} (target {a!r})")


**Expected output**

Output:

```text
'Rewrite the word river using only capital letters.': 'RULE OF FORCE' (target 'RIVER')
'What do you get when you add 47 to 4?': '51' (target '51')
'Name the word at the end of: wizard circle lemon feather cloud meadow': 'meadow' (target 'meadow')
'Tell me how many words this has: orange mirror tunnel butter rocket': '5' (target '5')
'Output the word ticket thrice.': 'ticket ticket' (target 'ticket ticket ticket')
'Tell me the initial letter of planet.': 'p' (target 'p')
```
